<a href="https://colab.research.google.com/github/SAfonso/paper-to-csv/blob/main/AthropicKeyTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 16.2 MB/s eta 0:00:00


In [ ]:
import os
import anthropic
import base64
import pandas as pd
from pathlib import Path
from google.colab import drive
from google.colab import userdata

In [ ]:
drive.mount('/content/drive')

RUTA_FOTOS      = '/content/drive/MyDrive/PapelesOncle/Imagenes'
RUTA_SALIDA_CSV = '/content/drive/MyDrive/PapelesOncle/Ficheros/contactos_final.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

In [ ]:
def extraer_datos_con_claude(ruta_imagen):
    client = anthropic.Anthropic()

    ext = Path(ruta_imagen).suffix.lower()
    media_type = "image/jpeg" if ext in [".jpg", ".jpeg"] else "image/png"

    with open(ruta_imagen, "rb") as f:
        img_b64 = base64.standard_b64encode(f.read()).decode("utf-8")

    mensaje = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": img_b64,
                    },
                },
                {
                    "type": "text",
                    "text": (
                        "Esta es una tarjeta de sorteo de Medianamente Comedy "
                        "con dos campos manuscritos: NOMBRE y Email.\n"
                        "Extrae exactamente el nombre y el email escritos a mano.\n\n"
                        "Reglas:\n"
                        "- Si el email no parece válido (no tiene @ y dominio), pon EMAIL: unknown\n"
                        "- Si el nombre está vacío o ilegible pero el email es válido, "
                        "intenta deducir un nombre a partir del email (ej: 'gorka.mujica@gmail.com' -> 'Gorka Mujica')\n"
                        "- Si no puedes leer algún campo, escribe ILEGIBLE.\n\n"
                        "Responde ÚNICAMENTE en este formato, sin nada más:\n"
                        "NOMBRE: <nombre>\n"
                        "EMAIL: <email>"
                    )
                }
            ],
        }],
    )

    respuesta = mensaje.content[0].text.strip()
    nombre, email = "No detectado", "No detectado"

    for linea in respuesta.split("\n"):
        if linea.startswith("NOMBRE:"):
            nombre = linea.replace("NOMBRE:", "").strip()
        elif linea.startswith("EMAIL:"):
            email = linea.replace("EMAIL:", "").strip()

    return nombre, email

In [ ]:
lista_contactos = []

archivos = sorted([
    f for f in os.listdir(RUTA_FOTOS)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f"Imágenes encontradas: {len(archivos)}\n")

for archivo in archivos:
    ruta = os.path.join(RUTA_FOTOS, archivo)
    try:
        nombre, email = extraer_datos_con_claude(ruta)
        print(f"✅ [{archivo}] -> {nombre} | {email}")
        lista_contactos.append({"archivo": archivo, "nombre": nombre, "email": email})
    except Exception as e:
        print(f"❌ [{archivo}] -> ERROR: {e}")
        lista_contactos.append({"archivo": archivo, "nombre": "ERROR", "email": str(e)})


Imágenes encontradas: 185

✅ [PXL_20260609_140102257.jpg] -> Alberto Jimenez | alberto.jimenez.tabasco@gmail.com
✅ [PXL_20260609_140107436.MP.jpg] -> Gema Cubero | gemcuco@gmail.com
✅ [PXL_20260609_140112780.jpg] -> Marisa Martinez | marisama1975@gmail.com
✅ [PXL_20260609_140119741.MP.jpg] -> Yolanda Adrià | yadria@gmail.com
✅ [PXL_20260609_140124839.jpg] -> Rafaela Calzado | villanueva.veronica@gmail.com
✅ [PXL_20260609_140129213.MP.jpg] -> Cristina Valdivielso | cris.valpa@gmail.com
✅ [PXL_20260609_140132937.MP.jpg] -> Veronica Villanueva | villanueva.veronica.c@gmail.com
✅ [PXL_20260609_140136908.jpg] -> Núria Vilarrasa | nuriag@bellvitgehospital.cat
✅ [PXL_20260609_140140568.jpg] -> Josep Mª | xeme64@gmail.com
✅ [PXL_20260609_140144421.jpg] -> Gala Roman | romangala513@gmail.com
✅ [PXL_20260609_143202861.jpg] -> Raul | raul.avensa@hotmail.com
✅ [PXL_20260609_143207890.jpg] -> Alexandra | alexandralg4444@gmail.com
✅ [PXL_20260609_143212950.jpg] -> Raquel Gómez | rgomezreina@gmail.co

In [ ]:
df = pd.DataFrame(lista_contactos)

# Deduplicar por email, quedándose con la primera aparición
# Los 'unknown' e 'ilegible' no se deduiplican entre sí
mask_validos = ~df['email'].str.lower().isin(['unknown', 'ilegible', 'no detectado'])
df_validos = df[mask_validos].drop_duplicates(subset='email', keep='first')
df_invalidos = df[~mask_validos]
df_final = pd.concat([df_validos, df_invalidos]).sort_index()

duplicados = len(df) - len(df_final)
if duplicados > 0:
    print(f"⚠️  {duplicados} duplicado(s) eliminado(s)")

df_final.to_csv(RUTA_SALIDA_CSV, index=False, encoding='utf-8')
print(f"\n✅ CSV guardado: {len(df_final)} contactos únicos")
df_final.head(10)